# Geomagnetic Rigidity Cutoff

Earth's magnetic field deflects cosmic rays below a location-dependent rigidity threshold $R_\text{cut}$ (~0 GV at the poles, ~15–17 GV near the equator). Because rigidity depends on charge $Z$, the cutoff affects nuclear species differently. We compare its effect on three GSF model types using $R_\text{cut} = 20$ GV (high, for visibility).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from globalsplinefit import GSFEnergy, GSFEnergyPerNucleon, GSFRigidity

plt.rcParams["figure.figsize"] = (10, 6)

R_CUT = 20.0  # GV
groups = [
    ("p", "r", "Proton"),
    ("He", "orange", "Helium"),
    ("O*", "g", "Oxygen*"),
    ("Fe*", "b", "Iron*"),
]

## Nucleus Flux (`GSFEnergy`)

At total energy $E$, a nucleus with charge $Z$ and mass $A$ has rigidity $R = \sqrt{E^2 - (A m_N)^2} / Z$. Protons ($A/Z = 1$) hit the cutoff at higher total energies than heavier nuclei ($A/Z \approx 2$). Left panel: absolute fluxes with (dashed) and without (solid) cutoff. Right panel: suppression ratio.

In [ ]:
energy = np.logspace(0, 3, 500)  # 1 GeV to 10 PeV
energy_exponent = 2.6
gsf = GSFEnergy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for name, color, label in groups:
    f0 = gsf.flux(energy, name)
    f1 = gsf.flux(energy, name, rigidity_cutoff=R_CUT)
    ax1.plot(
        energy, f0 * energy**energy_exponent, "-", color=color, lw=1.5, label=label
    )
    ax1.plot(energy, f1 * energy**energy_exponent, "--", color=color, lw=1.5)

f0_tot = gsf.total_flux(energy)
f1_tot = gsf.total_flux(energy, rigidity_cutoff=R_CUT)
ax1.plot(energy, f0_tot * energy**energy_exponent, "-", color="k", lw=2, label="Total")
ax1.plot(energy, f1_tot * energy**energy_exponent, "--", color="k", lw=2)

ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Energy [GeV]")
ax1.set_ylabel(f"Nucleus Flux x (E/GeV)$^{{{energy_exponent}}}$ [1/(GeV m² s sr)]")
ax1.set_xlim(energy[0], energy[-1])
ax1.set_ylim(1e0, 3e4)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title(f"Nucleus Flux (solid: no cutoff, dashed: $R_{{cut}}$ = {R_CUT} GV)")

# Ratio plot
for name, color, label in groups:
    f0 = gsf.flux(energy, name)
    f1 = gsf.flux(energy, name, rigidity_cutoff=R_CUT)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(f0 > 0, f1 / f0, np.nan)
    ax2.plot(energy, ratio, "-", color=color, lw=1.5, label=label)

ax2.set_xscale("log")
ax2.set_xlabel("Energy [GeV]")
ax2.set_ylabel("Flux ratio (with / without cutoff)")
ax2.set_xlim(energy[0], energy[-1])
ax2.set_ylim(-0.05, 1.15)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_title(f"Suppression by $R_{{cut}}$ = {R_CUT} GV")
ax2.axhline(1.0, color="gray", ls="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Rigidity Flux (`GSFRigidity`)

In rigidity space the cutoff is a sharp vertical boundary at $R_\text{cut}$, identical for all species.

In [ ]:
rigidity = np.logspace(-1, 4, 500)  # 0.1 GV to 10 TV
rig_exponent = 2.6
gsf_r = GSFRigidity()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for name, color, label in groups:
    f0 = gsf_r.flux(rigidity, name)
    f1 = gsf_r.flux(rigidity, name, rigidity_cutoff=R_CUT)
    ax1.plot(
        rigidity, f0 * rigidity**rig_exponent, "-", color=color, lw=1.5, label=label
    )
    ax1.plot(rigidity, f1 * rigidity**rig_exponent, "--", color=color, lw=1.5)

f0_tot = gsf_r.total_flux(rigidity)
f1_tot = gsf_r.total_flux(rigidity, rigidity_cutoff=R_CUT)
ax1.plot(rigidity, f0_tot * rigidity**rig_exponent, "-", color="k", lw=2, label="Total")
ax1.plot(rigidity, f1_tot * rigidity**rig_exponent, "--", color="k", lw=2)

ax1.axvline(
    R_CUT, color="gray", ls=":", lw=2, alpha=0.7, label=f"$R_{{cut}}$ = {R_CUT} GV"
)
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Rigidity [GV]")
ax1.set_ylabel(f"Rigidity Flux x (R/GV)$^{{{rig_exponent}}}$ [1/(GV m² s sr)]")
ax1.set_xlim(rigidity[0], rigidity[-1])
ax1.set_ylim(1e-1, 1e5)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title(f"Rigidity Flux (solid: no cutoff, dashed: $R_{{cut}}$ = {R_CUT} GV)")

# Ratio
for name, color, label in groups:
    f0 = gsf_r.flux(rigidity, name)
    f1 = gsf_r.flux(rigidity, name, rigidity_cutoff=R_CUT)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(f0 > 0, f1 / f0, np.nan)
    ax2.plot(rigidity, ratio, "-", color=color, lw=1.5, label=label)

ax2.axvline(
    R_CUT, color="gray", ls=":", lw=2, alpha=0.7, label=f"$R_{{cut}}$ = {R_CUT} GV"
)
ax2.set_xscale("log")
ax2.set_xlabel("Rigidity [GV]")
ax2.set_ylabel("Flux ratio (with / without cutoff)")
ax2.set_xlim(rigidity[0], rigidity[-1])
ax2.set_ylim(-0.05, 1.15)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_title(f"Sharp cutoff at $R_{{cut}}$ = {R_CUT} GV")
ax2.axhline(1.0, color="gray", ls="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Nucleon Flux (`GSFEnergyPerNucleon`)

The cutoff is applied per element during the nucleon-flux summation. At a given $E_N$, nucleus $i$ has rigidity $R_i = (A_i/Z_i) \sqrt{E_N^2 - m_N^2}$, so the cutoff in $E_N$ is roughly twice as high for protons as for heavier nuclei. Left: absolute flux with 1-sigma bands; uncertainties propagate through the cutoff via the Jacobian. Right: relative uncertainty with (dashed) and without (solid) cutoff.

In [ ]:
ekin = np.logspace(0, 3, 500)  # 1 GeV to 10 PeV per nucleon
en_exponent = 3.0
gsf_n = GSFEnergyPerNucleon()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for name, color, label in groups:
    f0 = gsf_n.flux(ekin, name)
    f1 = gsf_n.flux(ekin, name, rigidity_cutoff=R_CUT)
    e1 = gsf_n.error(ekin, name, rigidity_cutoff=R_CUT)

    axes[0].plot(ekin, f0 * ekin**en_exponent, "-", color=color, lw=1.5, label=label)
    axes[0].plot(ekin, f1 * ekin**en_exponent, "--", color=color, lw=1.5)
    axes[0].fill_between(
        ekin,
        (f1 - e1) * ekin**en_exponent,
        (f1 + e1) * ekin**en_exponent,
        facecolor=color,
        alpha=0.15,
    )

f0_tot = gsf_n.total_flux(ekin)
f1_tot = gsf_n.total_flux(ekin, rigidity_cutoff=R_CUT)
e1_tot = gsf_n.total_error(ekin, rigidity_cutoff=R_CUT)

axes[0].plot(ekin, f0_tot * ekin**en_exponent, "-", color="k", lw=2, label="Total")
axes[0].plot(ekin, f1_tot * ekin**en_exponent, "--", color="k", lw=2)
axes[0].fill_between(
    ekin,
    (f1_tot - e1_tot) * ekin**en_exponent,
    (f1_tot + e1_tot) * ekin**en_exponent,
    facecolor="k",
    alpha=0.1,
)

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"$E_N$ [GeV/nucleon]")
axes[0].set_ylabel(
    rf"Nucleon Flux x $(E_N/\mathrm{{GeV}})^{{{en_exponent}}}$ [1/(GeV m² s sr)]"
)
axes[0].set_xlim(ekin[0], ekin[-1])
axes[0].set_ylim(5e2, 1e5)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_title(f"Nucleon Flux (solid: no cutoff, dashed: $R_{{cut}}$ = {R_CUT} GV)")

# Relative error comparison
for name, color, label in groups:
    f0 = gsf_n.flux(ekin, name)
    e0 = gsf_n.error(ekin, name)
    f1 = gsf_n.flux(ekin, name, rigidity_cutoff=R_CUT)
    e1 = gsf_n.error(ekin, name, rigidity_cutoff=R_CUT)
    with np.errstate(divide="ignore", invalid="ignore"):
        re0 = e0 / np.maximum(f0, 1e-90)
        re1 = e1 / np.maximum(f1, 1e-90)
    valid = (f1 > 0) & np.isfinite(re1)
    axes[1].plot(ekin, re0, "-", color=color, lw=1.5, alpha=0.4)
    if np.any(valid):
        axes[1].plot(ekin[valid], re1[valid], "--", color=color, lw=1.5, label=label)

axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"$E_N$ [GeV/nucleon]")
axes[1].set_ylabel("Relative Uncertainty")
axes[1].set_xlim(ekin[0], ekin[-1])
axes[1].set_ylim(3e-3, 1)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_title("Relative Uncertainty (solid: no cutoff, dashed: with cutoff)")

plt.tight_layout()
plt.show()

## Summary

- In **rigidity space** the cutoff is a universal sharp threshold.
- In **energy space** the cutoff energy depends on $Z/A$; protons are suppressed up to higher energies than heavier nuclei.
- **Uncertainties propagate correctly** through the cutoff with no user intervention.
- The `rigidity_cutoff` parameter is available on all flux/error methods.